In [1]:
# Imports & Device 설정
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 디바이스:", device)


사용 디바이스: cuda


In [2]:
# 데이터 로드 (CIFAR-10)
transform = transforms.Compose([
    transforms.ToTensor(),  # (H,W,C) 0~255 -> (C,H,W) 0~1
])

train_set = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print(f"학습 데이터: {len(train_set)}장, 테스트 데이터: {len(test_set)}장")


100%|██████████| 170M/170M [22:06<00:00, 129kB/s]


학습 데이터: 50000장, 테스트 데이터: 10000장


In [3]:
# 모델 정의
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 32x32x32
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            # 16x16x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # 16x16x64
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            # 8x8x64

            nn.Conv2d(64, 64, kernel_size=3, padding=1),   # 8x8x64
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CNN(num_classes=10).to(device)
print(model)

# 파라미터 수 확인 (README 표에 넣을 값)
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=10, bias=True)
  )
)
전체 파라미터 수: 319,178


In [4]:
# 손실함수, 옵티마이저

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# 학습 루프
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


In [6]:
# 실행

EPOCHS = 10
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    elapsed = time.time() - start
    print(f"[Epoch {epoch+1}/{EPOCHS}] "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)")

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

[Epoch 1/10] train_loss=1.5540 train_acc=0.4325 val_loss=1.2836 val_acc=0.5428 (14.4s)
[Epoch 2/10] train_loss=1.1379 train_acc=0.5914 val_loss=1.0380 val_acc=0.6315 (11.4s)
[Epoch 3/10] train_loss=0.9698 train_acc=0.6575 val_loss=0.9644 val_acc=0.6563 (11.4s)
[Epoch 4/10] train_loss=0.8658 train_acc=0.6952 val_loss=0.8821 val_acc=0.6930 (11.3s)
[Epoch 5/10] train_loss=0.7895 train_acc=0.7227 val_loss=0.8750 val_acc=0.6902 (11.7s)
[Epoch 6/10] train_loss=0.7246 train_acc=0.7459 val_loss=0.8212 val_acc=0.7155 (10.7s)
[Epoch 7/10] train_loss=0.6673 train_acc=0.7667 val_loss=0.8018 val_acc=0.7261 (10.9s)
[Epoch 8/10] train_loss=0.6220 train_acc=0.7823 val_loss=0.8140 val_acc=0.7209 (12.5s)
[Epoch 9/10] train_loss=0.5789 train_acc=0.7961 val_loss=0.7723 val_acc=0.7363 (11.2s)
[Epoch 10/10] train_loss=0.5309 train_acc=0.8137 val_loss=0.7995 val_acc=0.7291 (11.1s)


In [7]:
# 모델 저장
torch.save(model.state_dict(), "cnn_pytorch.pth")
print("모델 저장 완료: cnn_pytorch.pth")

모델 저장 완료: cnn_pytorch.pth


In [15]:
# ONNX 변환 + 프레임워크 간 추론 비교를 위해 설치
!pip install onnx onnxruntime -q

In [9]:
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
import time

# 학습된 PyTorch 모델 불러오기
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = CNN(num_classes=10)
model.load_state_dict(torch.load("cnn_pytorch.pth", map_location="cpu"))
model.eval()

print("PyTorch 모델 로드 완료")

PyTorch 모델 로드 완료


In [10]:
# PyTorch -> ONNX 변환
dummy_input = torch.randn(1, 3, 32, 32)  # batch=1 예시 입력

torch.onnx.export(
    model,
    dummy_input,
    "cnn_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"},
    },
    opset_version=13,
    dynamo=False,  # PyTorch 2.9+ 에서는 새 dynamo 기반 exporter가 기본이라
                   # onnxscript 패키지가 추가로 필요함. 여기서는 기존 방식(legacy
                   # TorchScript 기반)을 그대로 써서 별도 설치 없이 동작하게 함.
)

print("ONNX 변환 완료: cnn_model.onnx")

# 변환된 모델이 유효한지 검사
onnx_model = onnx.load("cnn_model.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX 모델 유효성 검사 통과")

ONNX 변환 완료: cnn_model.onnx
ONNX 모델 유효성 검사 통과


/tmp/ipykernel_456/686674372.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [11]:
# ONNX Runtime으로 추론

session = ort.InferenceSession("cnn_model.onnx", providers=["CPUExecutionProvider"])

def predict_onnx(images_np):
    """
    images_np: (batch, 3, 32, 32), float32, 0~1 정규화된 numpy 배열
    """
    outputs = session.run(
        None,
        {"input": images_np.astype(np.float32)}
    )
    return outputs[0]  # logits


# 새 섹션

In [12]:
# 결과 검증: PyTorch 출력 vs ONNX 출력이 같은지 확인
test_input = torch.randn(8, 3, 32, 32)

with torch.no_grad():
    pytorch_out = model(test_input).numpy()

onnx_out = predict_onnx(test_input.numpy())

max_diff = np.max(np.abs(pytorch_out - onnx_out))
print(f"PyTorch vs ONNX 출력 최대 차이: {max_diff:.8f}")
print("(1e-4 이하면 사실상 동일한 결과로 볼 수 있음 -- 부동소수점 연산 순서 차이 정도)")


PyTorch vs ONNX 출력 최대 차이: 0.00001526
(1e-4 이하면 사실상 동일한 결과로 볼 수 있음 -- 부동소수점 연산 순서 차이 정도)


In [13]:
# 추론 속도 비교 (PyTorch vs ONNX Runtime)
def benchmark(fn, input_data, n_runs=50):
    # 워밍업
    for _ in range(5):
        fn(input_data)

    start = time.time()
    for _ in range(n_runs):
        fn(input_data)
    elapsed = time.time() - start

    return elapsed / n_runs  # 평균 추론 시간 (초)


batch_input = torch.randn(32, 3, 32, 32)
batch_input_np = batch_input.numpy()

def pytorch_infer(x):
    with torch.no_grad():
        return model(x)

pytorch_time = benchmark(pytorch_infer, batch_input)
onnx_time = benchmark(predict_onnx, batch_input_np)

print(f"\n=== 추론 속도 비교 (batch=32, {50}회 평균) ===")
print(f"PyTorch:      {pytorch_time*1000:.2f} ms/batch")
print(f"ONNX Runtime: {onnx_time*1000:.2f} ms/batch")
print(f"속도 차이:     {pytorch_time/onnx_time:.2f}x "
      f"({'ONNX가 더 빠름' if onnx_time < pytorch_time else 'PyTorch가 더 빠름'})")


=== 추론 속도 비교 (batch=32, 50회 평균) ===
PyTorch:      18.22 ms/batch
ONNX Runtime: 5.06 ms/batch
속도 차이:     3.60x (ONNX가 더 빠름)


In [14]:
# 정리하기 위한 딕셔너리

comparison_result = {
    "max_output_diff": float(max_diff),
    "pytorch_ms_per_batch": pytorch_time * 1000,
    "onnx_ms_per_batch": onnx_time * 1000,
}

print("\n비교 결과:", comparison_result)


비교 결과: {'max_output_diff': 1.52587890625e-05, 'pytorch_ms_per_batch': 18.215432167053223, 'onnx_ms_per_batch': 5.056796073913574}


In [16]:
from google.colab import files
files.download('cnn_pytorch.pth')
files.download('cnn_model.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>